# 🏃‍♀️💨 ADK 2 Orchestration — Your Marathon Coach Adventure! 🏅

Welcome, Agent Architect! 🎉 In this notebook you'll build a **Marathon Race Day Coach** and, one runnable rung at a time, master ADK 2's **three orchestration patterns**.

By the end of this adventure, you'll be able to:

- 🧩 **Compose graph workflows** — mix plain functions and agents as peers, fan out in parallel, and route with a plain `if`-statement.
- 🤝 **Coordinate collaborative agents** — let an LLM pick the *right* specialists and run them in parallel, and choose the right **mode** (`chat` / `task` / `single_turn`) for every subagent.
- 🌳 **Grow dynamic workflows** — let the LLM shape the work at runtime, with recursion kept safely bounded in code.

| Level | Idea |
|---|---|
| **P** 🎯 | Prologue — run the mega-prompt, watch it *invent* its data |
| **L0** 🐣 | `Agent` + `Runner` + a real **tool** — the atom |
| **L1** 🔗 | first `Workflow`: a function node and an agent node are peers |
| **L2a** 🌤️ | **Pillar 1a** — parallel fetch + `JoinNode` (one agent) |
| **L2b** 🚦 | **Pillar 1b** — add the deterministic router → 1 of 3 agents |
| **L3a** 🤝 | **Pillar 2** — same team, two worlds: `chat` strands the run; `single_turn` runs the subset in parallel |
| **L3b** 🎽 | **Pillar 2** — `task` mode: clarify → pause → resume → `finish_task` |
| **L4a** 🌱 | **Pillar 3a** — decompose → flat parallel research (runtime width) |
| **L4b** 🌳 | **Pillar 3b** — add recursive spawning (runtime depth) |
| **L5** 🧭 | which pattern, when (reading) |

**The through-line:** known structure → known team / variable subset → unknown shape → choose the right one 🎯

```
    ___       ___       ___       ___       ___
   |o o|     |^_^|     |•‿•|     |^o^|     |=^.^|
   |_-_|     |_-_|     |_-_|     |_-_|     |_-_|
    L0        L1       L2a·b     L3a·b     L4a·b·L5
   ready →   flows →   graph →   team →   dynamic →  🏁
```

**How each level works — three cells, one rhythm:** 📖 the *lesson* (short, every line is ADK) → 🔧 *run helpers* (folded; printing plumbing, not ADK — just run it, never required reading) → ▶ your *playground* (2 lines; edit, predict, re-run).

> 🏁 Run the cells **top to bottom**, including the folded 🔧 ones. Ready? Let's go! 👇

## Author ✍️

Hi, I'm **Qingyue (Annie) Wang**, a Developer Advocate and AI Engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)

If you have questions about this notebook, reach me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/), [X](https://twitter.com/anniewangtech), or email anniewangtech0510@gmail.com

```
  (\__/)
  (•ㅅ•)
  /づ  🏃   Enjoy building AI Agents — now go run your marathon! :)
```

---
## 🔑 Part 0 · Setup — pick your path

First things first — let's get your gear on! 🎽 Run the install cell below, then **one** of the two setup cells after it. Never both.

| | 🎓 **Workshop** | 🏠 **Take-home** |
| --- | --- | --- |
| **Who** | You're at a live workshop with a **credit claim link** | Everyone else — including workshop attendees, afterwards |
| **Runs on** | Vertex AI, on a project billed to your credit | Google AI Studio (free tier) |
| **Run** | the 🎓 cell | the 🏠 cell |

Everything from the Prologue onward is identical either way.

In [ ]:
# Pin the exact ADK 2 version this tutorial was verified on.
%pip install -q "google-adk==2.3.0" python-dotenv pydantic nest_asyncio
import nest_asyncio; nest_asyncio.apply()   # let Colab's running loop accept nested awaits
print("\u2713 installed")

---
### 🎓 Path A · Workshop (Google Cloud credit)

**Only if you're at a live workshop.** First claim your credit at the link your instructor shared (`https://me.developers.google.com/benefits/claim/…`) — using the **same Google account** you'll authorize below. Then run this cell and **skip Path B**.

It creates a project on your credit, enables the Vertex AI API, points the notebook at Vertex, and then calls Vertex once to make sure it actually answers before you go on.

> ⚠️ **Expect `waiting for Vertex AI to come up…` to print a couple of times.** That's normal — a project created seconds ago isn't allowed to serve yet, so the cell waits for it (up to 2 minutes).
>
> **And if any later cell ever fails with `403 … 'aiplatform.endpoints.predict' denied`** — that's the same waking-up window, *not* a billing or credit problem. Your project and credit are fine. Just **re-run this cell** and give it another minute.

In [1]:
# 🎓 WORKSHOP ONLY — run this INSTEAD of the AI Studio key cell below.
# Prereq: claim your credit first, using the SAME Google account you sign in with here.
import os, subprocess, sys, time

PROJECT_ID = ""            # leave blank to create one automatically
LOCATION   = "us-central1"
MODEL      = "gemini-2.5-flash"   # gemini-flash-latest is an AI-Studio-only alias

from google.colab import auth
auth.authenticate_user()                          # also authenticates the gcloud CLI
acct = subprocess.run(["gcloud", "auth", "list", "--filter=status:ACTIVE",
                       "--format=value(account)"],
                      capture_output=True, text=True).stdout.strip()
print(f"Signed in as: {acct}\n", flush=True)   # flush: Colab's stdout is not a tty,
                                               # so unflushed prints land AFTER subprocess output

if not PROJECT_ID:
    REPO = "/content/adk2-tutorial"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/cuppibla/adk2-tutorial.git", REPO], check=True)
    # Finds your GDP credit, creates adk-2-tutorial-XXXX, links it, writes ~/project_id.txt
    rc = subprocess.run([sys.executable, f"{REPO}/scripts/billing_enablement.py"]).returncode
    pid_file = os.path.expanduser("~/project_id.txt")
    if rc != 0 or not os.path.exists(pid_file):
        raise SystemExit(
            f"\n✋ Couldn't create the project automatically.\n"
            f"   Most likely the credit isn't claimed yet, or it was claimed on an\n"
            f"   account other than {acct}.\n"
            f"   → Claim it, wait ~30s, then re-run this cell.\n"
            f"   → Still stuck? Open https://shell.cloud.google.com and run:\n"
            f"        git clone https://github.com/cuppibla/adk2-tutorial.git\n"
            f"        cd adk2-tutorial && ./setup_billing.sh\n"
            f"     then paste the project ID into PROJECT_ID above and re-run this cell."
        )
    PROJECT_ID = open(pid_file).read().strip()

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True)
subprocess.run(["gcloud", "services", "enable",
                "aiplatform.googleapis.com", "--quiet"], check=True)

# Point ADK at Vertex AI instead of AI Studio.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"]      = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]     = LOCATION
os.environ["ADK_MODEL"]                 = MODEL
os.environ.pop("GOOGLE_API_KEY", None)                         # make sure no stale key wins
os.environ.pop("GEMINI_API_KEY", None)

# A brand-new project is NOT ready the moment gcloud returns. Both the API
# enablement and your owner IAM binding keep propagating for a minute or two,
# and a call inside that window fails with
#     403 Permission 'aiplatform.endpoints.predict' denied ... (or it may not exist)
# which reads like a broken setup and isn't. So prove the path works HERE,
# retrying, instead of letting the first level hit it.
from google import genai

WARMING = ("PERMISSION_DENIED", "SERVICE_DISABLED", "has not been used", "404")
probe = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
for attempt in range(1, 13):                      # up to ~2 min
    try:
        probe.models.generate_content(model=MODEL, contents="ping")
        break
    except Exception as e:
        warming = any(s in str(e) for s in WARMING)
        if attempt == 12 or not warming:
            hint = (f"   → Give it another minute and re-run this cell; new projects are\n"
                    f"     slow to wake. If it still fails, check billing is linked:\n"
                    f"        gcloud billing projects describe {PROJECT_ID}"
                    if warming else
                    "   → This isn't the usual propagation delay — read the error above.")
            raise SystemExit(
                f"\n✋ Vertex AI won't answer on {PROJECT_ID}.\n"
                f"   {type(e).__name__}: {str(e)[:300]}\n{hint}"
            )
        print(f"   waiting for Vertex AI to come up on the new project… ({attempt * 10}s)",
              flush=True)
        time.sleep(10)

print(f"\n✅ Vertex AI on {PROJECT_ID} · {LOCATION} · {MODEL} — answered a test call", flush=True)

MessageError: Error: credential propagation was unsuccessful

---
### 🏠 Path B · Take-home (AI Studio key)

**The default — and where workshop attendees come back to afterwards.** Get a free key at **[aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)** (click *Create API key*, copy it — starts with `AIza…`), then in Colab click the **🔑 Secrets** icon in the left sidebar → *Add new secret* → name it exactly **`GOOGLE_API_KEY`**, paste the value, toggle **Notebook access ON**.

Ran Path A already? **Skip this cell** — it would switch you back to AI Studio.

In [ ]:
import os

# 🏠 TAKE-HOME ONLY — if you ran the Workshop setup cell, skip this one.
if os.environ.get("GOOGLE_GENAI_USE_VERTEXAI") == "True":
    raise SystemExit("✋ You're set up on the workshop path (Vertex AI). Skip this cell.")

# Google AI Studio API key — add GOOGLE_API_KEY in the 🔑 Secrets panel (or paste when prompted).
try:
    from google.colab import userdata
    key = userdata.get("GOOGLE_API_KEY")
except Exception:
    import getpass
    key = getpass.getpass("Enter your Google AI Studio API key: ")

os.environ["GOOGLE_API_KEY"] = "".join(key.split())    # drop any stray whitespace/newlines
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"      # use AI Studio, not Vertex AI
print("✅ API key set — using Google AI Studio.")

---
## 📦 Shared building blocks

Structured I/O is how ADK 2 moves typed data between function nodes and agents — like passing a clean baton 🏃‍♀️➡️🏃. These Pydantic schemas + canned marathon scenarios are reused from L2 onward. **Run this cell once**, then keep going.

In [ ]:
# Generated from shared/schemas.py + shared/scenarios.py by notebooks/build.py.
# (No `from __future__ import annotations` — deferred string annotations break
#  pydantic forward-refs for nested models in a single notebook namespace.)
import os
MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")

from typing import Literal

from pydantic import BaseModel, Field


# ─── Pillar 1 (graph workflow) ────────────────────────────────────────────────

class WeatherData(BaseModel):
    temp_f: float
    wind_mph: float
    wind_direction: str
    humidity_pct: int
    conditions: str


class CourseData(BaseModel):
    name: str
    distance_mi: float
    total_elevation_gain_ft: int
    hardest_mile: int
    hardest_mile_grade_pct: float


class FitnessData(BaseModel):
    avg_pace_per_mile_sec: int
    longest_recent_run_mi: float
    weekly_mileage_mi: int


class RaceStrategy(BaseModel):
    target_finish: str = Field(
        description="Clean finish time only, format H:MM:SS. Example: '3:30:00'. "
        "Do NOT include explanations, parentheticals, or extra commentary."
    )
    pacing_advice: str = Field(description="One concise sentence about race pacing strategy.")
    fueling_plan: str = Field(description="One concise sentence about hydration and nutrition.")
    gear: str = Field(description="One concise sentence about clothing and equipment.")
    key_warning: str = Field(description="One concise sentence flagging the biggest risk for this race.")


class BundledRunData(BaseModel):
    """Shape of the JoinNode payload — keys match the upstream function names."""
    fetch_weather: WeatherData
    analyze_course: CourseData
    pull_fitness: FitnessData


# ─── Pillar 2 (collaborative agents) ──────────────────────────────────────────

class SpecialistInput(BaseModel):
    """Payload the concierge coordinator hands to each specialist subagent."""
    user_question: str = Field(description="The runner's free-form question.")
    current_strategy: RaceStrategy = Field(description="The strategy generated in L2.")
    runner_data: BundledRunData = Field(description="The original bundled weather/course/fitness data.")


class SpecialistResponse(BaseModel):
    """Structured response each specialist returns to the coordinator."""
    concern_level: Literal["none", "minor", "moderate", "serious"] = Field(
        description="Severity of the concern this specialist raises."
    )
    recommendation: str = Field(description="One concise sentence with the actionable recommendation.")
    reasoning: str = Field(description="One concise sentence with the why, referencing specific data.")


# ─── Pillar 3 (dynamic workflow) ──────────────────────────────────────────────

class DecomposerOutput(BaseModel):
    """Output of the decompose agent — the research plan."""
    plan_summary: str = Field(
        description="One sentence explaining how you decomposed the question."
    )
    sub_questions: list[str] = Field(
        description="3-7 specific, non-overlapping sub-questions that together "
                    "comprehensively answer the user's main question.",
        min_length=3,
        max_length=7,
    )


class ResearchFinding(BaseModel):
    """Output of one research agent invocation — findings on a single sub-question."""
    summary: str = Field(
        description="Concise 2-3 sentence summary of findings on this sub-question."
    )
    key_facts: list[str] = Field(
        description="3-5 specific factual points or actionable insights discovered.",
        min_length=2,
        max_length=5,
    )
    needs_deeper: bool = Field(
        description="True if these findings reveal a topic that warrants deeper "
                    "recursive investigation. Set False if the question is fully answered."
    )
    deeper_questions: list[str] = Field(
        default_factory=list,
        description="If needs_deeper is True, 1-3 specific, well-formed deeper "
                    "questions to investigate. Empty list if needs_deeper is False.",
        max_length=3,
    )


class DeepResearchBriefing(BaseModel):
    """Final synthesized output of the deep research workflow."""
    headline: str = Field(
        description="One-sentence headline summarizing the most important takeaway."
    )
    sections: list[str] = Field(
        description="3-6 thematic paragraphs covering the research areas.",
        min_length=3,
        max_length=6,
    )
    key_warnings: list[str] = Field(
        description="2-4 actionable warnings or critical considerations.",
        min_length=1,
        max_length=4,
    )
    summary: str = Field(
        description="Closing paragraph that ties everything together actionably."
    )

import os
from typing import Any



SCENARIOS: dict[str, dict[str, Any]] = {
    "HOT": {
        "weather": WeatherData(
            temp_f=78, wind_mph=12, wind_direction="headwind",
            humidity_pct=70, conditions="sunny",
        ),
        "course": CourseData(
            name="Boston Marathon", distance_mi=26.2,
            total_elevation_gain_ft=815, hardest_mile=20,
            hardest_mile_grade_pct=4.5,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=450,  # 7:30 / mile
            longest_recent_run_mi=22, weekly_mileage_mi=55,
        ),
    },
    "NORMAL": {
        "weather": WeatherData(
            temp_f=58, wind_mph=4, wind_direction="calm",
            humidity_pct=55, conditions="overcast",
        ),
        "course": CourseData(
            name="Berlin Marathon", distance_mi=26.2,
            total_elevation_gain_ft=240, hardest_mile=0,
            hardest_mile_grade_pct=0.5,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=420,  # 7:00 / mile
            longest_recent_run_mi=24, weekly_mileage_mi=65,
        ),
    },
    "COLD": {
        "weather": WeatherData(
            temp_f=35, wind_mph=15, wind_direction="crosswind",
            humidity_pct=65, conditions="cloudy",
        ),
        "course": CourseData(
            name="Chicago Marathon", distance_mi=26.2,
            total_elevation_gain_ft=140, hardest_mile=0,
            hardest_mile_grade_pct=0.3,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=440,  # 7:20 / mile
            longest_recent_run_mi=22, weekly_mileage_mi=60,
        ),
    },
}


def scenario() -> dict[str, Any]:
    """Return the currently selected scenario dict (HOT / NORMAL / COLD)."""
    name = os.environ.get("MARATHON_SCENARIO", "HOT").upper()
    if name not in SCENARIOS:
        raise ValueError(f"Unknown scenario {name!r}; expected HOT/NORMAL/COLD")
    return SCENARIOS[name]


def slow_mo() -> float:
    """Multiplier on simulated fetch latencies (env var MARATHON_SLOW_MO)."""
    try:
        return float(os.environ.get("MARATHON_SLOW_MO", "1.0"))
    except ValueError:
        return 1.0

print("\u2713 schemas + scenarios ready")

---
## 🎯 Prologue · Why Not One Big Prompt?

**⚡ Before you run it, lock in the ONE thing to watch: where does every specific number come from?** That's the entire exercise — everything else is decoration.

Before the ladder, run the thing the ladder replaces: **one agent whose prompt promises everything** — fetch the weather, analyze the course, read the training log, route by conditions, output the plan.

**What you'll see:** a confident, specific, well-formatted strategy… whose numbers are **invented**. In one live run it opened with *"I have pulled today's weather metrics"* and reported 52°F, a 9 mph wind, and an analysis of a training log it has never seen. There is no weather API here, no course data, no log — one opaque model call either fabricates its inputs or hedges them into uselessness.

That's the disease, and it has four symptoms worth naming:

1. **You can't trust it** — the data is made up, fluently.
2. **You can't test it** — step 4's routing lives inside prose; there's no `if` to unit-test.
3. **You can't swap a step** — no seam where a real weather API could plug in.
4. **You pay for everything, every time** — five steps, one giant call, no caching a deterministic part.

Hold that feeling. The next nine levels take those steps **out of the prompt, one at a time**: functions fetch (L1–L2a), an `if`-statement routes (L2b), specialists divide the work (L3a–L3b), and code bounds the shape (L4a–L4b).

In [ ]:
import asyncio
import os
import sys


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes



MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )

# The mega-prompt: one agent, promised everything. (Deliberately bad — this is
# the "before" picture.)
mega_coach = Agent(
    name="mega_coach",
    model=MODEL,
    instruction="""You are a complete marathon race-day strategy system. You have
access to today's weather forecast, the official course elevation data, and the
runner's training history. For any race the runner names, do ALL of the following:
1. Report today's race-day weather (temperature, wind, humidity).
2. Analyze the course's hardest mile and its grade.
3. Assess the runner's current fitness from their training log.
4. If it is hot, produce a heat strategy; if cold, a cold strategy; otherwise a
   normal-conditions strategy.
5. Output the full plan with SPECIFIC numbers throughout.""",
)


async def run(question: str = "Plan my race-day strategy for the Chicago Marathon.") -> None:
    runner = Runner(node=mega_coach, app_name="prologue",
                    session_service=InMemorySessionService(), auto_create_session=True)
    print(f"\n🏃 You: {question}")
    print(
        "\n🎯 WATCH ONE THING while it talks: every specific number below —\n"
        "   temperature, wind, grades, YOUR weekly mileage — ask where it came from.\n"
        "\n🤖 mega_coach:"
    )
    async for event in runner.run_async(
        user_id="u1", session_id="s1",
        new_message=gtypes.Content(role="user", parts=[gtypes.Part(text=question)]),
    ):
        for part in (getattr(getattr(event, "message", None), "parts", None) or []):
            if getattr(part, "text", None):
                print(part.text, end="", flush=True)
    print(
        "\n\n⚠️  Three questions before you scroll on:\n"
        "    1. Where did the temperature come from?   (there is no weather API here)\n"
        "    2. Where did the course grades come from? (there is no course data here)\n"
        "    3. Where did YOUR training log come from? (it has never seen one)\n"
        "    One opaque model call INVENTED its own inputs — fluently. You cannot\n"
        "    test its routing, swap in a real API, or trust a single number. The\n"
        "    next nine levels take these steps out of the prompt, one at a time.\n"
    )


def main() -> None:
    q = " ".join(sys.argv[1:]) or "Plan my race-day strategy for the Chicago Marathon."
    asyncio.run(run(q))

await run()

---
## 🐣 L0 · Your First Agent — the Pace Coach

Every marathon starts with one step 🐾 — a model, an instruction, and one REAL tool.

**⚡ TL;DR:** an agent is a model + an instruction + **tools it may call**; a `Runner` executes it. Everything after this level is just more agents, arranged in better shapes.

**The question:** can you get a model to answer — and to reach for **real code** when arithmetic matters?

**The one idea — three parts:**

- **`Agent`** — the thing that reasons (a Gemini model + an instruction).
- **`Runner`** — the thing that executes an agent inside a session and streams events.
- **a tool** — a plain Python function (`pace_splits`) the **model decides** to call. ADK reads the signature + docstring and hands the model a declaration; no schema-writing.

After the prologue this is the first repair: an LLM doing pace arithmetic in its head will happily be wrong — `pace_splits` is deterministic Python, so the numbers in the answer are **computed, not improvised**.

> 🔍 **The markers:** `Agent(...)` · `tools=[pace_splits]` · `Runner(...)`. And in the output, the `🔧` line — that's the **model deciding**, mid-answer, to call your code.

**What you'll see:**

```
   🔧 model called tool → pace_splits({'target_finish': '3:30:00'})
   🔧 tool returned     → {'per_mile': '8:00', 'per_km': '4:58', ...}
🧠 Coach: To finish in 3:30:00, you need an average pace of 8:00 per mile…
```

The 🔧 lines are the lesson: mid-answer, the **model chose** to call your function, and the exact `8:00/mile` in its reply came from your code — not from token statistics.

> 💡 Everything else in this codelab is just *more agents, arranged in more interesting shapes*. This is the atom.

> ❓ **You might be wondering:** *does the model always call the tool?* No — it decides, per question. Ask something with no numbers in it and the 🔧 lines vanish (the playground has you try exactly this).

> 👀 **Read:** `pace_splits` (a plain function) and the `tools=[pace_splits]` line. · ▶ **Run** it. · ✏️ **Change:** ask the *general* question (no goal time) — notice the 🔧 lines disappear: **the model decides** when a tool is worth calling. Then rewrite the `instruction` and re-run — the instruction is the rest of the program.

```
+---------------------------------------------+
|           🏃  pace_coach   (Agent)          |
|---------------------------------------------|
|  model : gemini-flash-latest                |
|  role  : friendly, concise marathon coach   |
|  runs  : Runner  >  Session  >  events      |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes




MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


# 1) A tool is just a Python function with a docstring and type hints —
#    ADK reads the signature and hands the model a declaration for it.
def pace_splits(target_finish: str) -> dict:
    """Convert a marathon goal time like '3:30:00' (or '3:30') into exact
    per-mile and per-km paces. Use this instead of estimating arithmetic."""
    parts = target_finish.strip().split(":")
    h, m, s = ([0, 0, 0] + [int(x) for x in parts])[-3:]
    total = h * 3600 + m * 60 + s
    fmt = lambda sec: f"{int(sec // 60)}:{int(sec % 60):02d}"
    return {
        "per_mile": fmt(total / 26.2188),
        "per_km": fmt(total / 42.195),
        "goal": f"{h}:{m:02d}:{s:02d}",
    }


# 2) Define an agent — a model, a role, and the tools it may use.
pace_coach = Agent(
    name="pace_coach",
    model=MODEL,
    tools=[pace_splits],   # ★ the model MAY call this — it decides, per question
    instruction=(
        "You are a friendly, concise marathon coach. Answer the runner's question "
        "in 3-4 sentences. Be specific and practical. No preamble. If the runner "
        "mentions a goal time, call pace_splits for exact paces — never do the "
        "arithmetic yourself."
    ),
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
def _event_text(event) -> str | None:
    """Pull human-readable text out of an event, whichever field carries it."""
    if getattr(event, "message", None) and getattr(event.message, "parts", None):
        chunks = [p.text for p in event.message.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def ask(question: str) -> None:
    # 2) A Runner executes the agent inside a session and streams back events.
    runner = Runner(
        node=pace_coach,
        app_name="l0_first_agent",
        session_service=InMemorySessionService(),
        auto_create_session=True,
    )

    # 3) Wrap the user's question as a Content message and run.
    message = gtypes.Content(role="user", parts=[gtypes.Part(text=question)])
    print(f"\n🏃 You: {question}\n")
    print("🧠 Coach: ", end="", flush=True)
    async for event in runner.run_async(
        user_id="runner_1",
        session_id="session_1",
        new_message=message,
    ):
        for part in (getattr(getattr(event, "message", None), "parts", None) or []):
            fc = getattr(part, "function_call", None)
            if fc:
                print(f"\n   🔧 model called tool → {fc.name}({dict(fc.args) if fc.args else ''})")
            fr = getattr(part, "function_response", None)
            if fr:
                print(f"   🔧 tool returned     → {fr.response}\n🧠 Coach: ", end="", flush=True)
        text = _event_text(event)
        if text:
            print(text, end="", flush=True)
    print("\n")


def main() -> None:
    question = " ".join(sys.argv[1:]) or "I want to finish in 3:30:00 — what pace do I need, and how should I run the first 5k?"
    asyncio.run(ask(question))

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await ask("I want to finish in 3:30:00 — what pace do I need?")   # watch the 🔧 tool call
await ask("What is the most common mistake first-time marathoners make?")  # no numbers → the model skips the tool

---
## 🔗 L1 · Your First Workflow — code meets model

Predictable work stays a function (0 LLM); only reasoning is an agent.

**⚡ TL;DR:** a plain function and an LLM agent are the **same kind of node**. Predictable work → function (0 LLM, deterministic); reasoning → agent.

**The question:** how do you mix plain code and an LLM in one flow, without paying for a model call on the parts that are just code?

**The one idea:** in a `Workflow`, a **plain Python function and an LLM agent are both just nodes** in the same `edges` list.

```
START ──► fetch_conditions (function, 0 LLM) ──► advise (agent, 1 LLM)
```

> 🔍 **The markers:** one edge tuple — `(START, fetch_conditions, advise)` — with a bare Python function sitting in the middle of it, and `input_schema=` validating the hand-off.

**What's new vs L0:** `Workflow(edges=[...])`, `START` (where input enters), a function node returning `Event(output=...)`, and `input_schema=Conditions` so the function's output is validated against that schema before the agent sees it (as JSON text — `input_schema` validates the boundary, it does not hand the agent a Python object).

> ❓ **You might be wondering:** *is function-then-agent the required order?* No — any order, any mix, any count. `advise` runs second only because it needs `fetch_conditions`' data. The lesson is the peerage, not the sequence.

> 👀 **Read:** `fetch_conditions` returns data with **no model call**; `advise` has `input_schema=Conditions`. · ▶ **Run** it. · ✏️ **Change:** set `temp_f=30` in the function and re-run — the advice flips, and the function still cost 0 LLM calls.

```
+---------------------------------------------+
|          🔗  l1_workflow  (Workflow)        |
|---------------------------------------------|
|  START > fetch_conditions > advise          |
|          (function, 0 LLM)   (agent, 1 LLM) |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys

from pydantic import BaseModel

from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import START
from google.adk.sessions import InMemorySessionService




MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


# ─── A tiny schema so the function node and the agent speak the same shape ─────

class Conditions(BaseModel):
    temp_f: float
    wind_mph: float
    conditions: str


# ─── Node 1: a plain function. No model, no cost — just prepares data. ─────────

def fetch_conditions(node_input):
    """In a real app this hits a weather API. Here it returns canned data.

    Returning `Event(output=...)` is the explicit form. A function node may also
    return a bare value or a pydantic model and ADK wraps it for you — the explicit
    form is used here because L2b needs its sibling, `Event(output=..., route=...)`.
    """
    data = Conditions(temp_f=78, wind_mph=12, conditions="sunny")
    print(f"  [fetch_conditions] (function node, 0 LLM) → {data.model_dump()}")
    return Event(output=data.model_dump())


# ─── Node 2: an agent. The only step that calls the model. ────────────────────

advise = Agent(
    name="advise",
    model=MODEL,
    input_schema=Conditions,  # receives the function node's output, validated
    instruction=(
        "You are a marathon coach. Given today's race-day conditions, give the "
        "runner 2-3 sentences of specific advice on pacing and gear for THESE "
        "conditions. Reference the actual temperature and wind."
    ),
)


# ─── The workflow: two nodes, one edge chain. ─────────────────────────────────

workflow = Workflow(
    name="l1_graph_basics",
    description="One function node feeds one agent node.",
    edges=[(START, fetch_conditions, advise)],   # ★ a function and an agent — peers in one chain
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
def _event_text(event) -> str | None:
    if getattr(event, "message", None) and getattr(event.message, "parts", None):
        chunks = [p.text for p in event.message.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def main() -> None:
    runner = Runner(
        node=workflow,
        app_name="l1_graph_basics",
        session_service=InMemorySessionService(),
        auto_create_session=True,
    )
    print("=== L1 workflow: fetch_conditions (function) → advise (agent) ===")
    async for event in runner.run_async(
        user_id="runner_1", session_id="session_1", new_message=None,
    ):
        text = _event_text(event)
        if text:
            print(f"\n🧠 Coach: {text}\n")

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await main()

---
## 🌤️ L2a · Parallel Fan-out + JoinNode (Pillar 1a)

Fan out, join, decide ✨ Try `run("COLD")` and watch the plan flip!

**⚡ TL;DR:** fan out in parallel (free), wait for **all**, bundle, hand one agent the complete picture.

**The question:** you can *draw the flow before the input arrives*. Start with the skeleton: gather data in parallel, bundle it, hand it to one agent.

**The shape:**

```
START ──► fetch_weather ──┐
START ──► analyze_course ─┼─► JoinNode ─► strategy (1 agent)
START ──► pull_fitness ───┘   (bundles)
```

> 🔍 **The markers:** three edges that all start at `START` — that *is* the fan-out — and `JoinNode`, the meeting point.

- The three fetches are **functions** — they run **in parallel**, 0 LLM calls.
- **`JoinNode`** waits for all three and bundles them into one typed payload (`BundledRunData`), keyed by function name.
- One `strategy` agent reads the bundle and writes a `RaceStrategy`.

**What you'll see:** each fetch prints a `started` / `finished` timestamp. All three start at **0.0s** and the fan-out ends at **2.0s** — the slowest fetch, not the **4.5s** their durations would sum to. That overlap is the parallelism. (The *total* wall time printed at the end is ~8s because it also contains the strategy agent's LLM call — read the fetch timestamps for the parallel claim, not the total.)

> 💡 **Where ADK 2 gives this a direct home:** function nodes and an agent node are peers in one `edges` list. 1.x could keep steps out of the model too — via a custom `BaseAgent` subclass — but that meant writing the orchestration plumbing yourself, so most builds wrapped each step as an agent.

> 💡 **Prologue callback:** the mega-prompt *invented* its weather. Here the temperature comes out of a fetch **function** — real code, real seam. Swap the canned dict for an actual weather API and nothing else changes.

> ❓ **You might be wondering:** *how much `JoinNode` do I need to understand?* One sentence: it waits until every parallel branch lands, packs the outputs into **one dict keyed by the upstream function's name**, and computes nothing itself. That dict is exactly why L2b's router can write `node_input["fetch_weather"]["temp_f"]`.

> 👀 **Read:** three edges fan out from `START`; `JoinNode` bundles them for one agent. · ▶ **Run** it and read the *timestamps*, not the total. · ✏️ **Change:** make one fetch sleep `3.0` — predict the new fan-out end time first, then verify.

```
+---------------------------------------------+
|       🌤️  l2a_parallel_join  (Workflow)     |
|---------------------------------------------|
|  fetch_weather  --+                          |
|  analyze_course --+--> JoinNode --> strategy |
|  pull_fitness   --+   (parallel,0LLM)   (1)  |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import JoinNode, START
from google.adk.sessions import InMemorySessionService





MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


# ─── Fetch nodes: parallel, zero LLM. Simulated latency so you SEE concurrency. ─
#
# Each fetch prints when it starts and finishes, relative to the start of the run.
# That overlap is the actual evidence for the parallel claim — all three start at
# ~0.0s and the fan-out ends when the SLOWEST one does, not when their durations
# add up. (Wall time for the whole run is a bad proxy: it also contains the
# strategy agent's LLM call, which is several seconds on its own.)

_T0 = 0.0


def _stamp(msg: str) -> None:
    print(f"  [t={time.perf_counter() - _T0:4.1f}s] {msg}")


async def _fetch(name: str, seconds: float, key: str):
    _stamp(f"{name} started")
    await asyncio.sleep(seconds * slow_mo())
    _stamp(f"{name} finished")
    return Event(output=scenario()[key].model_dump())


async def fetch_weather(node_input):
    return await _fetch("fetch_weather", 1.5, "weather")


async def analyze_course(node_input):
    return await _fetch("analyze_course", 2.0, "course")


async def pull_fitness(node_input):
    return await _fetch("pull_fitness", 1.0, "fitness")


# ─── Join: bundle the three parallel results into one typed payload. ──────────

join_inputs = JoinNode(name="join_inputs")   # ★ waits for ALL branches, bundles outputs keyed by function name


# ─── One strategy agent (no routing yet). ─────────────────────────────────────

strategy = Agent(
    name="strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction="""You are a marathon coach. You receive BundledRunData (weather from
fetch_weather, course from analyze_course, fitness from pull_fitness). Produce a
RaceStrategy that ADAPTS to whatever the conditions are — hot, cold, or ideal.
Cite actual numbers (temps, mile numbers, the runner's pace). Each field 1-2
short sentences.""",
)


# ─── The workflow: fan-out, join, one agent. ──────────────────────────────────

root = Workflow(
    name="l2a_parallel_join",
    description="Parallel data gathering + a single strategy agent.",
    edges=[
        # ★ three edges from START — this IS the fan-out
        (START, fetch_weather, join_inputs),
        (START, analyze_course, join_inputs),
        (START, pull_fitness, join_inputs),
        (join_inputs, strategy),
    ],
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
def _event_text(event):
    msg = getattr(event, "message", None)
    if msg and getattr(msg, "parts", None):
        chunks = [p.text for p in msg.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def run(scenario_name="HOT"):
    global _T0
    os.environ["MARATHON_SCENARIO"] = scenario_name.upper()
    print(f"=== L2a · scenario: {scenario_name.upper()} ===")
    runner = Runner(node=root, app_name="l2a", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = _T0 = time.perf_counter()
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=None):
        t = _event_text(event)
        if t:
            print(f"\n🏁 RaceStrategy:\n{t}")
    print(f"\n  Total wall time: {time.perf_counter()-t0:.1f}s "
          f"(fan-out + join + 1 LLM call — the fetch timestamps above are the parallel evidence)")

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await run("HOT")

---
## 🚦 L2b · Add the Deterministic Router (Pillar 1b)

One `if`-statement instead of a model decision 💸 🔁 Try `run("NORMAL")` too.

**⚡ TL;DR:** L2a untouched + a plain `if` decides which **one** agent runs. Branching, without asking the model.

**The question:** the plan should differ for hot vs cold weather. How do you branch — *without* asking the model to decide?

**The shape (L2a + a router):**

```
… JoinNode ─► route_by_weather ─► hot_strategy
               (if-statement)   ─► normal_strategy
                                ─► cold_strategy
```

> 🔍 **The markers:** `Event(output=…, route=…)` — a function node *naming* the path — and the dict-edge `{"HOT": …, "NORMAL": …, "COLD": …}` that maps names to nodes.

**The takeaway — three kinds of work, three homes:**

- Predictable work → **functions** (the 3 parallel fetches)
- A clear rule → **explicit routing** (`route_by_weather` is an `if`-statement, not a model decision)
- Reasoning → **the model** (exactly **one** strategy agent runs)

**What you'll see:** `temp=78F -> route=HOT`, then a structured `RaceStrategy`. **Net cost: 1 LLM call.**

> 💡 The common 1.x build wrapped each step as an agent — **4 calls** instead of this pattern's **1**. (A custom `BaseAgent` subclass could reach 1 call in 1.x as well; it just wasn't first-class, so few builds did it.)

> ⚠️ **If you add a fourth branch,** give the route-dict a `DEFAULT_ROUTE` entry too. A route the dict doesn't match isn't an error — the branch simply ends, and the program exits **0 with no output**, which is a confusing dead end to debug.

> ❓ **You might be wondering:** *so L2b is literally L2a plus a router?* Yes — the fetches and the join are untouched, and it's still exactly **1 LLM call**. What changed: "always the same agent" became "one of three, chosen by data".

> 👀 **Read:** `route_by_weather` — the router is an `if`-statement, not an agent. · ▶ **Run** `run("COLD")` too. · ✏️ **Change:** add a `WINDY` branch with a fourth agent — and read the `DEFAULT_ROUTE` warning above *before* you do.

```
+---------------------------------------------+
|      🚦  marathon_strategy  (Workflow)      |
|---------------------------------------------|
|  3 parallel fetches > JoinNode > router     |
|  > hot / normal / cold      (1 LLM call)    |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import JoinNode, START
from google.adk.sessions import InMemorySessionService





MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


# ─── Fetch nodes (same as L2a): parallel, zero LLM. ───────────────────────────

async def fetch_weather(node_input):
    await asyncio.sleep(1.5 * slow_mo())
    return Event(output=scenario()["weather"].model_dump())


async def analyze_course(node_input):
    await asyncio.sleep(2.0 * slow_mo())
    return Event(output=scenario()["course"].model_dump())


async def pull_fitness(node_input):
    await asyncio.sleep(1.0 * slow_mo())
    return Event(output=scenario()["fitness"].model_dump())


join_inputs = JoinNode(name="join_inputs")


# ─── The new part: a deterministic router. An if-statement, not the model. ────

def route_by_weather(node_input):
    """node_input is the JoinNode payload, keyed by upstream function names.
    Emit `route=` to pick exactly one downstream branch."""
    temp = node_input["fetch_weather"]["temp_f"]
    if temp >= 70:
        route = "HOT"
    elif temp <= 40:
        route = "COLD"
    else:
        route = "NORMAL"
    print(f"  [router] temp={temp}°F → route={route}  (an if-statement, 0 LLM)")
    return Event(output=node_input, route=route)   # ★ the function NAMES the path


# ─── Three specialized strategy agents. Only one ever runs. ───────────────────

_FMT = """
You receive BundledRunData (weather/course/fitness). Produce a RaceStrategy.
Cite actual numbers (temps, mile numbers, the runner's pace). Each field 1-2
short sentences.
"""

hot_strategy = Agent(
    name="hot_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in HOT conditions.
Heat is the primary risk: slow down, hydrate aggressively, dress cool, set a
goal time SLOWER than ideal.{_FMT}""",
)

normal_strategy = Agent(
    name="normal_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in IDEAL conditions.
This is a PR-attempt day. Recommend even or slightly negative splits; the main
risk is going out too fast on cool, fast-feeling pavement.{_FMT}""",
)

cold_strategy = Agent(
    name="cold_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in COLD conditions.
Recommend layered, shed-able gear, a careful warm-up, and fueling that accounts
for cold-suppressed thirst. If winds are strong, advise drafting.{_FMT}""",
)


# ─── The workflow: L2a + the router and its dict-edge. ────────────────────────

root = Workflow(
    name="marathon_strategy",
    description="Parallel data gathering + deterministic routing to one of three agents.",
    edges=[
        (START, fetch_weather, join_inputs),
        (START, analyze_course, join_inputs),
        (START, pull_fitness, join_inputs),
        (join_inputs, route_by_weather),
        (route_by_weather, {
            # ★ dict-edge: route name → node. Add a branch? Add a DEFAULT_ROUTE too.
            "HOT": hot_strategy,
            "NORMAL": normal_strategy,
            "COLD": cold_strategy,
        }),
    ],
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
def _event_text(event):
    msg = getattr(event, "message", None)
    if msg and getattr(msg, "parts", None):
        chunks = [p.text for p in msg.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def run(scenario_name="HOT"):
    os.environ["MARATHON_SCENARIO"] = scenario_name.upper()
    print(f"=== L2b · scenario: {scenario_name.upper()} ===")
    runner = Runner(node=root, app_name="l2b", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=None):
        t = _event_text(event)
        if t:
            print(f"\n🏁 RaceStrategy:\n{t}")
    print(f"\n  Wall time: {time.perf_counter()-t0:.1f}s (3 fetches parallel; strategy = 1 LLM call)")

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await run("HOT")

---
## 🤝 L3a · Collaborative Agents: One Flag, Two Worlds (Pillar 2)

Same team, run twice — the only diff is one flag. 💰 Beat 1 ≈ 2 LLM calls · Beat 2 ≈ 4–8 (depends on the subset).

**⚡ TL;DR:** same team, one flag. `chat` hands the **whole conversation** to one specialist and never comes back; `single_turn` turns each specialist into a **tool** — parallel subset, auto-return, one synthesis.

**The question:** you know the **team**, but the **request** decides which members should answer. How do you let an LLM pick the subset — and run them concurrently?

**The shape:** a coordinator over six specialists (medical, weather, pacing, gear, nutrition, mental). This level runs the **same team twice** — same coordinator prompt, same six specialists. The only difference is one flag on the subagents. **The contrast is the lesson.**

> 🔍 **The markers:** `mode="single_turn"` in the factory — and in the *output*, `TRANSFER →` (beat 1) versus a burst of `DISPATCH →` lines sharing one timestamp (beat 2).

### Beat 1 · Run the default first — and watch it fail the job

No `mode=` written → subagents default to **`chat`**. What you'll see:

```
TRANSFER → nutrition_specialist   (transfer_to_agent — the only tool chat subagents provide)
Final speaker: nutrition_specialist
```

The coordinator got **no delegation tools** — chat subagents only give it `transfer_to_agent`, a serial handoff of the *whole conversation* to **one** specialist. That specialist answers the user directly, and the run ends there. No parallel dispatch. No return. No synthesis. Ask the broad question and it gets worse: six specialists, one transfer.

That's not a bug — it's chat mode doing its job. The conversation *belongs* to whoever holds it, until someone explicitly transfers away. Right for an open-ended assistant; wrong for a pipeline step.

### Beat 2 · One flag, two worlds

The only diff: `mode="single_turn"` on each specialist. Same question, run again:

```
[t= 7.8s] DISPATCH → medical_specialist      ← same timestamp =
[t= 7.8s] DISPATCH → weather_specialist         one turn, many calls
[t=14.5s]   ↩ medical_specialist replied     ← replies land inside
[t=14.5s]   ↩ weather_specialist replied        one short window
🧠 Concierge (synthesized): <one answer>
```

Now ADK injects **one delegation tool per specialist** — named after the subagent, described by its `description=` (that text is what the coordinator reads when choosing the subset; skip it and you're routing on names alone). The coordinator emits several calls in one turn, ADK runs them **in parallel**, each auto-returns its result, and the coordinator synthesizes.

| Question | Specialists that fire |
| --- | --- |
| "What about fueling?" | nutrition only |
| "My knee hurts at mile 18" | medical only |
| "Should I race today?" | medical + weather + pacing |
| "Anything I should worry about?" | all 6 |

**Why each specialist gets handed the whole briefing:** each `single_turn` subagent runs in its **own isolated session branch** — it cannot see the conversation or its peers. Nothing is ambient: the coordinator must forward the *entire* `SpecialistInput` (question + strategy + runner data) separately into every parallel call.

> 💡 **Where ADK 2 gives this a direct home:** an LLM picks a **per-request subset** AND runs it in parallel — *declared* via `sub_agents` + `mode="single_turn"`. You could assemble the same shape in 1.x by wrapping each specialist in `AgentTool`; what changes is that it is now a declaration rather than plumbing. (`ParallelAgent` is always-all and `transfer_to_agent` is serial.)

> ⚠️ Two honest caveats: (1) the model picks the subset, so it's **less deterministic** than L2's hard-coded router — the exact subset can vary run to run. (2) Occasionally you'll see an `Error validating input: ...` line for one specialist. It is almost never the specialist's *output* — `output_schema` makes Gemini enforce that server-side. It's the **input**: the coordinator has to reproduce the whole nested `SpecialistInput` verbatim for every parallel call, and sometimes it fumbles one. ADK returns the error as that tool's result, the coordinator recovers, and the synthesis still lands.

> ❓ **You might be wondering:** *is `chat` just 1.x-style delegation — one agent at a time?* Essentially yes: it's the 1.x default behavior, now with a name. The gap to `single_turn` is three-dimensional: what the coordinator holds (one `transfer_to_agent` vs one tool **per specialist**) · how many can work (one, owning the conversation vs N in parallel) · whether control returns (never vs automatically, with results). *And about the code:* the factory's `if mode ==` branch exists **only** so one team can be built both ways for this contrast — a real app hardcodes one mode and the `if` disappears.

> 👀 **Read:** the `_specialist` factory — the `mode` parameter is the whole level. · ▶ **Run** both beats. · ✏️ **Change:** ask *"my knee hurts at mile 18"* — **predict the subset first**, then check the DISPATCH lines.

```
+-----------------------------------------------+
|       🤝  race_concierge  (coordinator)       |
|-----------------------------------------------|
|  6 specialists · same team, run twice:        |
|  chat (default)  > TRANSFER > stranded 🫠     |
|  single_turn     > parallel subset > 1 answer |
+-----------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )

SPECIALIST_SPECS = [
    ("medical_specialist", "medical", "injury risk, pain, when to stop, hydration safety, heat stroke"),
    ("weather_specialist", "weather", "heat, cold, wind, rain, race-day forecast adjustments"),
    ("pacing_specialist", "pacing", "pace strategy, mile splits, heart rate, target finish time"),
    ("gear_specialist", "gear", "clothing, shoes, accessories (hats, glasses, gloves), drop-bag contents"),
    ("nutrition_specialist", "nutrition", "fueling plan, gels, electrolytes, hydration timing, pre-race meals"),
    ("mental_specialist", "mental", "race mindset, motivation, pre-race anxiety, mid-race low points"),
]

SPECIALIST_NAMES = {name for name, _, _ in SPECIALIST_SPECS}


# ─── Specialist factory. `mode` is THE variable this level teaches. ────────────

def _specialist(name: str, domain: str, focus: str, mode: str | None) -> Agent:
    # NOTE: this `if` exists ONLY so one team can be built both ways for the
    # contrast — a real app hardcodes one mode and the branch disappears.
    kwargs = {}
    if mode == "single_turn":
        # The structured contract only makes sense when the specialist is a TOOL:
        # it receives a SpecialistInput and must return a SpecialistResponse.
        # A chat specialist talks to the user in prose — no schemas.
        kwargs = dict(mode="single_turn",   # ★ THE flag this level is about
                      input_schema=SpecialistInput, output_schema=SpecialistResponse)
    return Agent(
        name=name, model=MODEL,
        # `description` is NOT optional decoration: in single_turn mode ADK turns
        # each subagent into a tool and uses this text as that tool's description,
        # so it is what the coordinator actually reads when choosing the subset.
        description=f"Marathon {domain} specialist. Consult for: {focus}.",
        instruction=f"""You are a marathon {domain} specialist. Answer ONLY questions
within your domain. Focus: {focus}

Given the question, the current race strategy, and the runner's data, produce:
- concern_level: "none" | "minor" | "moderate" | "serious"
- recommendation: ONE concrete actionable sentence
- reasoning: ONE sentence citing specific numbers from the strategy or runner data

If the question is outside your domain, set concern_level="none" and say so briefly.""",
        **kwargs,
    )


def build_team(mode: str | None) -> Agent:
    """Build a FRESH team (agents can't be shared between parents).

    The coordinator's instruction is IDENTICAL in both modes — on purpose. In chat
    mode it physically cannot comply ("call the tools in parallel" — there are no
    tools), which is exactly the point: the capability lives in the mode flag, not
    in the prompt.
    """
    team = [_specialist(n, d, f, mode) for n, d, f in SPECIALIST_SPECS]
    return Agent(
        name="race_concierge",
        model=MODEL,
        sub_agents=team,
        instruction="""You are a marathon race day concierge. The runner already has a
strategy and is asking a follow-up. Six specialists are available:
medical, weather, pacing, gear, nutrition, mental.

For each question:
1. DECIDE which specialists are genuinely relevant — be precise. Examples:
   "My knee hurts" → medical only. "What about fueling?" → nutrition only.
   "Should I race today?" → medical + weather + pacing. "Anything I should worry
   about?" → all 6. Do NOT invoke specialists whose domain doesn't apply.
2. Call the relevant specialist tools IN PARALLEL (multiple function calls in one
   turn). Each takes a SpecialistInput: user_question (verbatim), current_strategy
   and runner_data (forward both from the user's initial message).
3. SYNTHESIZE their responses into one answer, under 4 sentences, leading with the
   most important concern.

The user's message contains the strategy + runner data as JSON — extract and forward them.""",
    )


# For `adk web` and imports: the version that actually does the job.
race_concierge = build_team("single_turn")

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
# ─── A run harness that shows WHO decided, WHO ran, and WHO answered. ──────────

def _build_message(question: str) -> gtypes.Content:
    """Give the coordinator a question plus a (canned) strategy + runner data."""
    s = scenario()
    runner_data = BundledRunData(
        fetch_weather=s["weather"], analyze_course=s["course"], pull_fitness=s["fitness"],
    )
    # Derive the strategy from the SAME scenario the runner data came from.
    # (Hardcoding a heat-stroke warning here would contradict the 35°F numbers a
    # learner sees after `MARATHON_SCENARIO=COLD` — the specialists would be
    # handed a self-contradicting brief.)
    w = s["weather"]
    if w.temp_f >= 70:
        pacing = "Start 20s/mile slower than goal pace to bank against the heat."
        gear = "Light singlet, cap, sunglasses."
        warning = (f"{w.temp_f:.0f}°F + {w.humidity_pct}% humidity — "
                   "real heat-stroke risk at your usual 7:30 pace.")
    elif w.temp_f <= 40:
        pacing = "Hold goal pace; the cold masks effort, so do not start too fast."
        gear = "Long sleeves, gloves, throwaway layer for the corral."
        warning = (f"{w.temp_f:.0f}°F — hypothermia risk if you slow down late; "
                   "keep a dry layer at the finish.")
    else:
        pacing = "Even splits — conditions are close to ideal for goal pace."
        gear = "Singlet and shorts; no weather adjustment needed."
        warning = (f"{w.temp_f:.0f}°F and {w.conditions} — no weather red flags; "
                   "the risk is going out too fast.")
    strategy = RaceStrategy(
        target_finish="3:32:00",
        pacing_advice=pacing,
        fueling_plan="Electrolytes every aid station.",
        gear=gear,
        key_warning=warning,
    )
    body = (
        f"{question}\n\n"
        f"--- context ---\n"
        f"current_strategy: {strategy.model_dump_json()}\n"
        f"runner_data: {runner_data.model_dump_json()}"
    )
    return gtypes.Content(role="user", parts=[gtypes.Part(text=body)])


async def ask(question: str, mode: str = "single_turn") -> None:
    label = "chat (the default — no mode= written)" if mode != "single_turn" else 'mode="single_turn"'
    coordinator = build_team(mode if mode == "single_turn" else None)
    runner = Runner(
        node=coordinator, app_name="l3a_concierge",
        session_service=InMemorySessionService(), auto_create_session=True,
    )
    print(f"\n━━ Subagents in {label} ━━")
    print(f"💬 Question: {question}\n")
    dispatched: list[str] = []
    returned: list[str] = []
    final_text, final_author = "", None
    t0 = time.perf_counter()
    async for event in runner.run_async(
        user_id="runner_1", session_id="s1", new_message=_build_message(question),
    ):
        author = getattr(event, "author", None)
        msg = getattr(event, "message", None)
        for part in getattr(msg, "parts", None) or []:
            fc = getattr(part, "function_call", None)
            # chat mode: the ONLY tool the coordinator has is transfer_to_agent —
            # a serial handoff of the whole conversation to one subagent.
            if fc and fc.name == "transfer_to_agent":
                target = (fc.args or {}).get("agent_name", "?")
                print(f"  [t={time.perf_counter()-t0:4.1f}s] TRANSFER → {target}"
                      "   (transfer_to_agent — the only tool chat subagents provide)")
            # single_turn mode: one delegation tool PER specialist, many calls in
            # one turn = the parallel dispatch.
            if fc and fc.name in SPECIALIST_NAMES and fc.name not in dispatched:
                dispatched.append(fc.name)
                print(f"  [t={time.perf_counter()-t0:4.1f}s] DISPATCH → {fc.name}")
            # Replies landing inside one short window = the parallel evidence.
            fr = getattr(part, "function_response", None)
            if fr and fr.name in SPECIALIST_NAMES and fr.name not in returned:
                returned.append(fr.name)
                print(f"  [t={time.perf_counter()-t0:4.1f}s]   ↩ {fr.name} replied")
            # Track WHO produced the last text — in chat mode it will be the
            # specialist (the conversation was carried away); in single_turn mode
            # it must be the coordinator's synthesis.
            if getattr(part, "text", None):
                final_text, final_author = part.text, author
    print(f"\n  Total time: {time.perf_counter()-t0:.1f}s")
    if mode == "single_turn":
        print(f"  Specialists chosen: {dispatched or ['(none)']}")
        if final_author == "race_concierge" and final_text:
            print(f"\n🧠 Concierge (synthesized):\n{final_text}\n")
        else:
            print("\n⚠️ No coordinator synthesis captured this run (rare) — re-run.")
    else:
        print(f"  Final speaker: {final_author}")
        if final_author != "race_concierge":
            print(f"\n🗣️ {final_author} (answering the user DIRECTLY):\n{final_text}")
            print(
                "\n⛔ Note what did NOT happen: no parallel dispatch, no return to the\n"
                "   coordinator, no synthesis. The conversation now BELONGS to this\n"
                "   specialist until someone transfers it away. That's chat mode —\n"
                "   right for an open-ended assistant, wrong for a pipeline step.\n"
            )
        else:
            print(f"\n🧠 Coordinator answered without delegating:\n{final_text}\n")


def main() -> None:
    args = [a for a in sys.argv[1:]]
    mode = "single_turn"
    if "--mode" in args:
        i = args.index("--mode")
        mode = args[i + 1] if i + 1 < len(args) else "single_turn"
        del args[i:i + 2]
    question = " ".join(args) or "Should I race today?"
    asyncio.run(ask(question, mode=mode))

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await ask("What about fueling?", mode="chat")   # Beat 1 — watch it strand
await ask("Should I race today?")               # Beat 2 — one flag, two worlds

---
## 🎽 L3b · Task Mode: a Conversation with a Finish Line (Pillar 2)

Clarify ➜ pause ➜ resume ➜ `finish_task` 🏁 (~4 LLM calls across two scripted turns).

**⚡ TL;DR:** the middle mode — talk to the user **until the fields are collected**, then auto-return with a **validated object**.

**The question:** L3a left a gap. `chat` owns the whole conversation; `single_turn` never talks to the user at all. But real intake work sits in between: *"talk to the user UNTIL you've collected X — then come back with a validated object."* Which mode is that?

**The shape:**

```
race_desk (coordinator)
  └─ gear_fitter (mode="task", output_schema=GearOrder)
```

> 🔍 **The markers:** `mode="task"` + `output_schema=` on the *same* agent — and in the output, the ⏸ pause and the `finish_task` call.

**What you'll see:**

```
━━ TURN 1 ━━  user: 'I need shoes for the marathon.'
  race_desk → delegate: gear_fitter
  gear_fitter: What is your shoe size?
  ⏸  The run ENDED — but nothing failed. This is a PAUSED task.

━━ TURN 2 ━━  user: 'Size 9, wide.'   (same session → resumes the task)
  gear_fitter → finish_task   (payload validates as GearOrder)
  race_desk: Your order ... in size 9 Wide has been confirmed.
```

Three things happened that neither L3a mode can do:

1. **The run genuinely stopped mid-task** — a *paused* task, not a hang and not a failure. The agent asked its clarifying question and is holding the task open. (In `adk web` you'd just type the answer; the harness scripts it as a second message on the same session.)
2. **The next message resumed the SAME task agent** — no re-routing, no re-delegation. The session knows who was waiting.
3. **`finish_task` ended it** — a tool ADK injected *because* of `mode="task"`. The agent must call it to finish, and its payload must validate against `output_schema`. A conversation with a **typed finish line** — then control auto-returns to the coordinator, result attached.

### The one-question rule for picking a mode

> 💡 **"Does the user need to talk to it — and until WHEN?"** chat = indefinitely · task = until the fields are collected · single_turn = never.

| Mode | Human in the loop | Parallel? | Returns to parent |
| --- | --- | --- | --- |
| `chat` *(subagent default)* — support assistant, open-ended copilot | full conversation | no | manual (via transfer) |
| `task` — intake, booking, troubleshooting | clarifying questions only | no | automatic (via `finish_task`, with a validated object) |
| `single_turn` — classify · extract · judge · generate | none | **yes** | automatic (with its result) |

`mode` goes on **subagents only** — never on the coordinator. And workflow *nodes* default to `single_turn` (which is why L1–L2b never wrote it), while *subagents* default to `chat` (which is why L3a had to).

> ⚠️ Two version notes before you build on this: (1) **`task` as a static graph node is version-dependent** — on 2.0.0b1–2.3.0 (this codelab's pin), `Workflow(...)` raises at construction; use exactly what this level does (a chat coordinator with task sub-agents) or dispatch via `ctx.run_node`. **Lifted in 2.5.0.** (2) **"Task agents must be leaf agents"** (no subagents of their own) is a documented ADK limitation — but a *contract*, not a runtime guard: neither 2.3.0 nor 2.5.0 will stop you. Don't read the absence of an error as permission.

> 💡 **Go deeper:** a `task` agent embedded in a *graph workflow* (the 2.5.0+ shape), with routing that can loop the conversation back for a retry: companion repo [`22_agent_in_workflow`](https://github.com/cuppibla/adk-workflows-compared/tree/main/examples/22_agent_in_workflow) · full mode guide: [`docs/agent-modes.md`](https://github.com/cuppibla/adk-workflows-compared/blob/main/docs/agent-modes.md).

> ❓ **You might be wondering:** *what does `task` buy me that the other two can't?* Three things: **auto-return** (chat carries the conversation away instead) · a **typed finish line** (`finish_task`'s payload must validate against the schema — you get data back, not a transcript) · **pause/resume** (the ⏸ is a held task waiting for a human, not a hang).

> 👀 **Read:** `gear_fitter` — `mode="task"` + `output_schema` is the entire contract. · ▶ **Run** it. · ✏️ **Change:** `run_desk("I need a hydration vest", "2 liters, medium")` — the clarifying question adapts, the finish line stays typed.

```
+-----------------------------------------------+
|          🎽  race_desk  (coordinator)         |
|-----------------------------------------------|
|  gear_fitter (mode="task", output_schema)     |
|  ask > ⏸ paused > scripted reply > resume >   |
|  finish_task(GearOrder) > auto-return  🏁     |
+-----------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys
import time


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes
from pydantic import BaseModel, Field




MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


class GearOrder(BaseModel):
    """The task's contract: the conversation may not end until this validates."""

    item: str = Field(description="the recommended piece of race gear")
    size: str = Field(description="the runner's size, incl. width/volume if given")
    note: str = Field(description="one-line fitting note")


gear_fitter = Agent(
    name="gear_fitter",
    model=MODEL,
    mode="task",                      # ★ the whole level, in one flag
    output_schema=GearOrder,          # ★ the typed finish line the conversation must reach
    description="Race-gear fitting desk. Collects the runner's size, then returns a GearOrder.",
    instruction="""You fit race gear. You MUST know the runner's size before finishing.
If the size is not given, ask ONE short clarifying question for it.
Once you have the size, finish the task with a GearOrder (pick a sensible item).""",
)

race_desk = Agent(
    name="race_desk",
    model=MODEL,
    sub_agents=[gear_fitter],
    instruction="""You are the race-expo front desk. For any gear or equipment request,
delegate to gear_fitter. When it returns its GearOrder, confirm the order back to
the runner in ONE sentence.""",
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
# ─── Harness: two turns, annotated. ────────────────────────────────────────────

async def _turn(runner: Runner, text: str, t0: float) -> tuple[str | None, bool]:
    """Run one user turn; return (author_of_last_text, finish_task_seen)."""
    msg = gtypes.Content(role="user", parts=[gtypes.Part(text=text)])
    last_author, finished = None, False
    async for event in runner.run_async(user_id="runner_1", session_id="s1", new_message=msg):
        author = getattr(event, "author", None)
        for part in (getattr(getattr(event, "message", None), "parts", None) or []):
            fc = getattr(part, "function_call", None)
            if fc and fc.name == "finish_task":
                finished = True
                print(f"  [t={time.perf_counter()-t0:4.1f}s] {author} → finish_task"
                      "   (the tool mode=\"task\" injected; payload validates as GearOrder)")
            elif fc:
                print(f"  [t={time.perf_counter()-t0:4.1f}s] {author} → delegate: {fc.name}")
            if getattr(part, "text", None):
                print(f"  [t={time.perf_counter()-t0:4.1f}s] {author}: {part.text.strip()}")
                last_author = author
    return last_author, finished


async def run_desk(question: str, scripted_reply: str) -> None:
    runner = Runner(
        node=race_desk, app_name="l3b_desk",
        session_service=InMemorySessionService(), auto_create_session=True,
    )
    t0 = time.perf_counter()

    print(f"\n━━ TURN 1 ━━  user: {question!r}")
    author, finished = await _turn(runner, question, t0)
    if not finished and author == "gear_fitter":
        print(
            "\n  ⏸  The run ENDED — but nothing failed. This is a PAUSED task:\n"
            "     gear_fitter asked its clarifying question and is holding the task\n"
            "     open, waiting for the user. (In `adk web` you would just type the\n"
            "     answer; here the next turn scripts it.)"
        )

    print(f"\n━━ TURN 2 ━━  user: {scripted_reply!r}   (same session → resumes the task)")
    author, finished = await _turn(runner, scripted_reply, t0)

    print(f"\n  Total time: {time.perf_counter()-t0:.1f}s")
    if finished and author == "race_desk":
        print(
            "  ✓ finish_task fired → the validated GearOrder flowed back to race_desk\n"
            "    automatically, and the coordinator spoke last. Automatic return is the\n"
            "    difference from chat mode — nobody had to transfer control back.\n"
        )
    else:
        print("  ⚠️ Unexpected shape this run (live LLM) — re-run; the flow above is the normal case.\n")


def main() -> None:
    args = list(sys.argv[1:])
    reply = "Size 9, wide."
    if "--reply" in args:
        i = args.index("--reply")
        reply = args[i + 1] if i + 1 < len(args) else reply
        del args[i:i + 2]
    question = " ".join(args) or "I need shoes for the marathon."
    asyncio.run(run_desk(question, reply))

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await run_desk("I need shoes for the marathon.", "Size 9, wide.")

---
## 🌱 L4a · Runtime-Sized Fan-out — Deep Research (Pillar 3a)

The LLM picks how MANY — the width is decided at runtime.

**⚡ TL;DR:** the skeleton is still three static steps — dynamic hides **inside** the middle one, where the width is decided by data at runtime.

> ⚠️ **Heads-up: this is the steepest step of the ladder.** The previous level was 44 lines; this one is ~120 — three agents and two workflow nodes, and none of it is padding. Budget ~15 minutes, and lean on the Read/Run/Change line at the end: you don't need to absorb every line on the first pass.

**The question:** the *shape* of the work depends on the input. You can't draw the graph ahead of time. Start with runtime **width**: let the LLM decide *how many* sub-questions.

**The shape (one level deep):**

```
START ─► decompose ─► research_topic (parallel_worker) ─► synthesize
                             │  │  │
                             └──┴──┴─ (flat: no children yet)
```

An open-ended question is **decomposed** into N sub-questions — **N is chosen by the LLM at runtime** (3–7) — each **researched in parallel**, then **synthesized** into one briefing.

> ⚠️ This cell makes **5–9 live LLM calls** (1 decompose + 3–7 research + 1 synthesize) and takes **~20–30s**. It costs real API quota.

> 🔍 **The markers — there is no `dynamic=True` switch.** Dynamic is a way of *writing*, not a config. Two markers and only two: `@node(parallel_worker=True)` (takes a runtime-sized list, runs one worker per item) and `ctx.run_node(...)` (code scheduling nodes directly). See either one → you're in dynamic.

**What you'll see:** the decomposer prints e.g. 5 sub-questions, they research in parallel, then a synthesized briefing. The *number* differs on every run — the fixed graph couldn't do that.

> 💡 **Where ADK 2 gives this a direct home:** `@node(parallel_worker=True)` fans one worker across a **runtime-sized** list. A 1.x `ParallelAgent` needs a fixed list known at build time; raw `asyncio` could size it at runtime, but then it is no longer a workflow ADK can trace.

**Two flags on the worker worth understanding:**

- **`rerun_on_resume=True` is mandatory** on any node that calls `ctx.run_node` — ADK raises a `ValueError` without it. On resume it must re-execute the dispatching node to rebuild the children it spawned, since those aren't in the static graph.
- **`retry_config=` bounds how this FAILS.** A parallel worker cancels every sibling and re-raises the instant one child fails — so without a retry, a single transient 429 discards the whole run, including every call already paid for. The retry lands on the inner per-item node, so each branch retries independently.

> ❓ **You might be wondering:** *where does ADK "know" this is dynamic?* It doesn't need to — nothing is declared anywhere. The decomposer produces a list at runtime; the parallel worker sizes itself to whatever arrives. The dynamism is a property of the data flow you wrote, not a mode you switched on.

> 👀 **Read:** the two flags on `research_topic` — `parallel_worker` and `rerun_on_resume`. · ▶ **Run** it. · ✏️ **Change:** swap in your own open question — N changes because the *input* decided the width.

```
+---------------------------------------------+
|      🌱  l4a_flat_research  (Workflow)       |
|---------------------------------------------|
|  decompose > research x N (parallel) >      |
|  synthesize     (width chosen at runtime)   |
+---------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import RetryConfig, START, node
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )


# ─── Three single-turn agents: decompose, research, synthesize. ───────────────

decompose_agent = Agent(
    name="decompose_agent", model=MODEL,
    output_schema=DecomposerOutput,
    instruction="""You are a research coordinator for marathon/endurance questions.
Break the user's open-ended question into 3-7 specific, non-overlapping,
independently-researchable sub-questions, each covering a distinct angle.""",
)

research_agent = Agent(
    name="research_agent", model=MODEL,
    output_schema=ResearchFinding,
    instruction="""You are a marathon research specialist. Given ONE specific
research question, produce a finding: a 2-3 sentence summary and 3-5 specific
insights. (For this level, keep needs_deeper=False.)""",
)

synthesize_agent = Agent(
    name="synthesize_agent", model=MODEL,
    output_schema=DeepResearchBriefing,
    instruction="""You are a marathon coach synthesizing a JSON list of findings
into one briefing for the runner: a HEADLINE, 3-6 thematic SECTIONS with specific
facts, 2-4 KEY_WARNINGS, and a closing SUMMARY. Write for the runner, be direct.""",
)


def _coerce(payload, schema_cls):
    if isinstance(payload, schema_cls):
        return payload
    if isinstance(payload, dict):
        return schema_cls.model_validate(payload)
    if isinstance(payload, str):
        return schema_cls.model_validate_json(payload)
    if hasattr(payload, "parts") and payload.parts:
        text = getattr(payload.parts[0], "text", None)
        if text:
            return schema_cls.model_validate_json(text)
    raise ValueError(f"Cannot coerce {type(payload).__name__} into {schema_cls.__name__}")


def _extract_text(node_input):
    if isinstance(node_input, str):
        return node_input
    if hasattr(node_input, "parts") and node_input.parts:
        return getattr(node_input.parts[0], "text", None) or str(node_input)
    return str(node_input)


# ─── Node 1: decompose into a runtime-sized list of sub-questions. ────────────

@node(rerun_on_resume=True)
async def decompose(ctx, node_input):
    user_query = _extract_text(node_input)
    plan = _coerce(await ctx.run_node(decompose_agent, node_input=user_query), DecomposerOutput)
    print(f"  [decompose] {len(plan.sub_questions)} sub-questions (width chosen at runtime):")
    for q in plan.sub_questions:
        print(f"    • {q[:80]}")
    yield Event(output=[{"question": q, "original_query": user_query} for q in plan.sub_questions])


# ─── Node 2: parallel_worker — one research task per item. Flat (no recursion). ─

# ─── Bounding the RATE, not just the shape. ───────────────────────────────────
#
# The fan-out width is already bounded by the schema (max_length=7). This bounds
# how it FAILS: a parallel worker cancels every sibling and re-raises the moment
# one child raises, so without a retry a single transient 429 throws away a whole
# run — including every call you already paid for. `retry_config` is applied to
# the INNER per-item node, so each branch retries on its own and a transient blip
# is absorbed before it can take the others down.
#
# Note `max_concurrency` is a real field on the parallel worker but is NOT
# reachable through the public `node()` API in ADK 2.3.0 — so on a rate-limited
# key, the schema bound is what keeps the in-flight count sane.
RESEARCH_RETRY = RetryConfig(max_attempts=3, initial_delay=2.0, backoff_factor=2.0)

# ★ parallel_worker: one worker per list item — and the list's SIZE arrives at runtime.
#   There is no dynamic=True switch anywhere; this flag and ctx.run_node ARE dynamic.
@node(parallel_worker=True, rerun_on_resume=True,
      retry_config=RESEARCH_RETRY)
async def research_topic(ctx, node_input):
    question = node_input["question"]
    ctxq = node_input.get("original_query", "")
    print(f"  [research] {question[:70]}")
    finding = _coerce(await ctx.run_node(
        research_agent,
        node_input=f"ORIGINAL QUERY: {ctxq}\n\nRESEARCH QUESTION: {question}",
    ), ResearchFinding)
    yield Event(output={"question": question, "summary": finding.summary, "key_facts": finding.key_facts})


# ─── Node 3: synthesize the flat list of findings. ────────────────────────────

@node(rerun_on_resume=True)
async def synthesize(ctx, node_input):
    print(f"  [synthesize] merging {len(node_input)} findings")
    briefing = _coerce(await ctx.run_node(
        synthesize_agent, node_input=json.dumps(node_input, indent=2)), DeepResearchBriefing)
    yield Event(output={"briefing": briefing.model_dump(), "findings": node_input})


l4a_workflow = Workflow(
    name="l4a_flat_research",
    description="Decompose (runtime width) → flat parallel research → synthesize.",
    edges=[(START, decompose, research_topic, synthesize)],
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
async def run(query="Tell me everything I should know about racing the Boston Marathon."):
    print(f"=== L4a · flat research ===\n  QUERY: {query}\n")
    runner = Runner(node=l4a_workflow, app_name="l4a", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    final = None
    async for event in runner.run_async(
        user_id="u1", session_id="s1",
        new_message=gtypes.Content(role="user", parts=[gtypes.Part(text=query)]),
    ):
        out = getattr(event, "output", None)
        if isinstance(out, dict) and "briefing" in out:
            final = out
    if final:
        b = final["briefing"]
        print(f"\n  {len(final['findings'])} sub-questions researched in parallel | {time.perf_counter()-t0:.1f}s")
        print(f"\n📋 {b['headline']}\n")
        for i, s in enumerate(b["sections"], 1):
            print(f"  {i}. {s[:180]}")

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await run()

---
## 🌳 L4b · Add Recursive Spawning (Pillar 3b)

…and how DEEP — bounded by `MAX_DEPTH`, in code. 🌲

**⚡ TL;DR:** recursion is **written, not given** — the worker calls *itself* through `ctx.run_node`, ordinary Python — so the brake must be written too. That's `MAX_DEPTH`.

**The question:** one research finding sometimes surfaces a narrow sub-topic worth its own investigation. How do you let a branch **spawn more parallel work** — and keep it bounded?

**The shape (now recursive):**

```
START ─► decompose ─► research_topic (parallel_worker, recursive) ─► synthesize
                             │  │  │
                             │  │  └─ research(q3) ─► maybe spawn children
                             │  └─── research(q2) ─► maybe spawn children
                             └────── research(q1) ─► maybe spawn children
```

> ⚠️ This cell makes **5–30 live LLM calls** and takes **20–45s** — the ceiling is 1 decompose + 7 top-level + 7×3 children + 1 synthesize. Run it deliberately.

> 🔍 **The markers:** `ctx.run_node(research_topic, …)` **inside** `research_topic` itself — self-reference *is* the recursion — and the guard `depth < MAX_DEPTH` one line above it.

**What you'll see:** research nodes printing `spawning N deeper` — recursion happening live — then a runtime tree shape (e.g. `5 top-level + 10 recursive children`). The tree differs on every run.

> 💡 **Where ADK 2 gives this a direct home:** runtime-sized *and* runtime-deep parallel fan-out with recursive `ctx.run_node`, all **inside the framework** — you keep tracing, checkpointing, and resumability. 1.x could recurse too, but only by dropping out to raw `asyncio`, which lost you all of that.

> 💡 **The rule:** *let the LLM shape the work, but keep the boundaries in code.* `MAX_DEPTH = 2` means depth-2 children cannot spawn — there is no depth 3. Width is bounded too (3–7 sub-questions).

> ⚠️ **Before you raise the knob:** the ceiling grows fast — `MAX_DEPTH=3` takes the worst case from ~30 calls to ~93. And at the very end of a run you may see a `cancelling N leftover tasks` log line: that's ADK tearing down its parallel task group after the result is already complete. Harmless — and depending on your logging config you may never see it.

> ❓ **You might be wondering:** *isn't dynamic recursive by default?* No — L4a is fully dynamic with **zero** recursion. Dynamic only hands you ordinary Python control flow; L4b *chooses* to write recursion with it. And because **you** wrote the recursion, **you** must write its boundary — this is where *"let the LLM shape the work, keep the boundaries in code"* stops being a slogan.

> 👀 **Read:** the guard: `if finding.needs_deeper and depth < MAX_DEPTH`. · ▶ **Run** it. · ✏️ **Change:** set `MAX_DEPTH = 1` and re-run — the tree flattens (and the run gets cheaper). The boundary is YOURS, in code.

```
+---------------------------------------------+
|        🌳  deep_research  (Workflow)         |
|---------------------------------------------|
|  decompose > research > (recurse: children) |
|  > synthesize    MAX_DEPTH = 2  guard-rail  |
+---------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import RetryConfig, START, node
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = os.getenv("ADK_MODEL", "gemini-flash-latest")   # AI Studio alias; on Vertex set ADK_MODEL=gemini-2.5-flash
if os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in ("true", "1", "yes") and "latest" in MODEL:
    sys.exit(
        f"\u2717 ADK_MODEL={MODEL!r} is an AI-Studio-only alias and 404s on Vertex AI.\n"
        "  export ADK_MODEL=gemini-2.5-flash   (then re-run)"
    )

# The boundary kept in CODE. The LLM decides width and depth, but never past this.
MAX_DEPTH = 2


decompose_agent = Agent(
    name="decompose_agent", model=MODEL,
    output_schema=DecomposerOutput,
    instruction="""You are a research coordinator for marathon/endurance questions.
Break the user's open-ended question into 3-7 specific, non-overlapping,
independently-researchable sub-questions, each covering a distinct angle.""",
)

research_agent = Agent(
    name="research_agent", model=MODEL,
    output_schema=ResearchFinding,
    instruction="""You are a marathon research specialist. Given ONE specific
research question, produce a 2-3 sentence summary and 3-5 insights. Set
needs_deeper=True ONLY if your findings surface a genuinely narrow technical
sub-topic that deserves its own investigation; then give 1-3 specific
deeper_questions.""",
)

synthesize_agent = Agent(
    name="synthesize_agent", model=MODEL,
    output_schema=DeepResearchBriefing,
    instruction="""You are a marathon coach synthesizing a nested JSON tree of
findings into one briefing: a HEADLINE, 3-6 SECTIONS combining related findings
with specific facts, 2-4 KEY_WARNINGS, and a closing SUMMARY. Write for the runner.""",
)


def _coerce(payload, schema_cls):
    if isinstance(payload, schema_cls):
        return payload
    if isinstance(payload, dict):
        return schema_cls.model_validate(payload)
    if isinstance(payload, str):
        return schema_cls.model_validate_json(payload)
    if hasattr(payload, "parts") and payload.parts:
        text = getattr(payload.parts[0], "text", None)
        if text:
            return schema_cls.model_validate_json(text)
    raise ValueError(f"Cannot coerce {type(payload).__name__} into {schema_cls.__name__}")


def _extract_text(node_input):
    if isinstance(node_input, str):
        return node_input
    if hasattr(node_input, "parts") and node_input.parts:
        return getattr(node_input.parts[0], "text", None) or str(node_input)
    return str(node_input)


@node(rerun_on_resume=True)
async def decompose(ctx, node_input):
    user_query = _extract_text(node_input)
    plan = _coerce(await ctx.run_node(decompose_agent, node_input=user_query), DecomposerOutput)
    print(f"  [decompose] {len(plan.sub_questions)} sub-questions (width chosen at runtime):")
    for q in plan.sub_questions:
        print(f"    • {q[:80]}")
    yield Event(output=[{"question": q, "depth": 1, "original_query": user_query} for q in plan.sub_questions])


# ─── Bounding the RATE, not just the shape. ───────────────────────────────────
#
# The fan-out width is already bounded by the schema (max_length=7). This bounds
# how it FAILS: a parallel worker cancels every sibling and re-raises the moment
# one child raises, so without a retry a single transient 429 throws away a whole
# run — including every call you already paid for. `retry_config` is applied to
# the INNER per-item node, so each branch retries on its own and a transient blip
# is absorbed before it can take the others down.
#
# Note `max_concurrency` is a real field on the parallel worker but is NOT
# reachable through the public `node()` API in ADK 2.3.0 — so on a rate-limited
# key, the schema bound is what keeps the in-flight count sane.
RESEARCH_RETRY = RetryConfig(max_attempts=3, initial_delay=2.0, backoff_factor=2.0)

@node(parallel_worker=True, rerun_on_resume=True,
      retry_config=RESEARCH_RETRY)
async def research_topic(ctx, node_input):
    question = node_input["question"]
    depth = node_input["depth"]
    ctxq = node_input.get("original_query", "")
    print(f"  [research d={depth}] {question[:66]}")
    finding = _coerce(await ctx.run_node(
        research_agent,
        node_input=f"ORIGINAL QUERY: {ctxq}\n\nRESEARCH QUESTION: {question}",
    ), ResearchFinding)

    children = []
    # ★ the brake — YOU wrote the recursion below, so YOU write its boundary.
    if finding.needs_deeper and finding.deeper_questions and depth < MAX_DEPTH:
        deeper = [{"question": dq, "depth": depth + 1, "original_query": ctxq}
                  for dq in finding.deeper_questions]
        print(f"  [research d={depth}] spawning {len(deeper)} deeper (depth chosen at runtime)")
        children = await ctx.run_node(research_topic, node_input=deeper)   # ★ calls ITSELF — ordinary Python recursion, no framework magic

    yield Event(output={"question": question, "depth": depth, "summary": finding.summary,
                        "key_facts": finding.key_facts, "children": children})


@node(rerun_on_resume=True)
async def synthesize(ctx, node_input):
    print(f"  [synthesize] merging {len(node_input)} top-level findings + their children")
    briefing = _coerce(await ctx.run_node(
        synthesize_agent, node_input=json.dumps(node_input, indent=2)), DeepResearchBriefing)
    yield Event(output={"briefing": briefing.model_dump(), "research_tree": node_input})


l4b_workflow = Workflow(
    name="deep_research",
    description="Decompose → recursive parallel research → synthesize.",
    edges=[(START, decompose, research_topic, synthesize)],
)

In [ ]:
#@title 🔧 Run helpers — printing only, NOT ADK. Run me once; expand only if curious.
async def run(query="Tell me everything I should know about racing the Boston Marathon."):
    print(f"=== L4b · recursive deep research ===\n  QUERY: {query}\n")
    runner = Runner(node=l4b_workflow, app_name="l4b", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    final = None
    async for event in runner.run_async(
        user_id="u1", session_id="s1",
        new_message=gtypes.Content(role="user", parts=[gtypes.Part(text=query)]),
    ):
        out = getattr(event, "output", None)
        if isinstance(out, dict) and "briefing" in out:
            final = out
    if final:
        tree = final["research_tree"]
        top = len(tree)
        deeper = sum(len(n.get("children", [])) for n in tree)
        b = final["briefing"]
        print(f"\n  Tree (runtime-decided): {top} top-level + {deeper} recursive children | {time.perf_counter()-t0:.1f}s")
        print(f"\n📋 {b['headline']}\n")
        for i, s in enumerate(b["sections"], 1):
            print(f"  {i}. {s[:180]}")

In [ ]:
# ▶ Your playground — edit these lines and re-run!
await run()

---
## 🧭 L5 · Which Pattern, When? (the finish line 🏁)

**⚡ TL;DR:** one axis decides everything — **who picks the next step: the graph you drew, the LLM, or your code.**

You've built all three. This is the model that makes them useful: **match the pattern to the shape of your problem.**

### The axis: who decides what runs next?

| Pillar | Who decides what runs next | Built in |
| --- | --- | --- |
| **1 · Graph** | the graph you drew | L2a / L2b |
| **2 · Collaborative** | the LLM | L3a / L3b |
| **3 · Dynamic** | your Python code, at runtime | L4a / L4b |

### Step 0: do you even need a graph?

ADK ships **prebuilt workflow agents** — `SequentialAgent`, `ParallelAgent`, `LoopAgent`. For a plain chain of agents, those are the cheapest correct answer and there's no graph to assemble. Reach past them when you need **explicit routing** (L2b's router), a **join** (L2a's `JoinNode`), or **nodes that aren't agents** (a plain function, zero LLM calls) — that last one is usually the reason.

```
Would a prebuilt SequentialAgent / ParallelAgent / LoopAgent do?
│
├─ YES ──────────────────────────────► use it; stop here
│
└─ NO — I need routing, a join, or non-agent nodes
   │
   Can you draw the workflow before the input arrives?
   │
   ├─ YES ───────────────────────────► Pillar 1 · Graph workflow    (L2a/L2b)
   │
   └─ NO
      ├─ Known team, request picks the subset? ─► Pillar 2 · Collaborative  (L3a/L3b)
      └─ Does the shape depend on the input?  ──► Pillar 3 · Dynamic        (L4a/L4b)
```

> ⚠️ **What this codelab did not teach you.** Nine rungs, ~50 minutes — the scope is deliberate. **Loops** (generate → review → fix in a `while`) are the canonical dynamic shape and aren't here; neither is graph-workflow **human input** (`RequestInput` — L3b's paused *task* is the collaborative cousin, not the graph node); and L4b's **resumability** is asserted but never demonstrated. All of it is covered in the companion repo [**adk-workflows-compared**](https://github.com/cuppibla/adk-workflows-compared) — see [`07_loop`](https://github.com/cuppibla/adk-workflows-compared/tree/main/examples/07_loop), [`17_request_input`](https://github.com/cuppibla/adk-workflows-compared/tree/main/examples/17_request_input), and [`docs/three-pillars.md`](https://github.com/cuppibla/adk-workflows-compared/blob/main/docs/three-pillars.md).

### The honest 1.x-vs-2 framing

This is **not** "2.0 can do things 1.x couldn't" — 1.x could build all of it. The shift is that **2.0 gives each shape a more direct home**, so known control flow leaves the prompt and becomes structure you can see and test.

| Pattern | The 1.x cost | The ADK 2 home |
| --- | --- | --- |
| **Graph** | 4 LLM calls in the common build; routing hidden in a prompt | function + agent nodes as peers → 1 call, `if`-statement router |
| **Collaborative** | buildable via `AgentTool` plumbing; `ParallelAgent` always-all, `transfer_to_agent` serial | a **declared** team: `sub_agents` + `mode="single_turn"` |
| **Dynamic** | recursion drops you out of the framework | `parallel_worker` + recursive `ctx.run_node` inside the framework |

### The whole app, and what a graph can't show you

You have now built every piece below. `Workflow` exposes its structure at `graph.edges`, so this picture is **generated from the code** rather than drawn by hand — and what the introspection finds *is* the summary of this lab:

| Pillar | What `graph.edges` contains | Why |
| --- | --- | --- |
| **1 · Graph** (L2b) | **10 edges**, routes and all | you drew it before any input arrived |
| **2 · Collaborative** (L3a) | **0 edges** — only `sub_agents` + `mode` | the LLM picks the subset per request |
| **3 · Dynamic** (L4a/L4b) | **3 edges — identical in both** | the recursion is written in Python, not wired in the graph |

That last row is the proof for the question L4b answers: L4a and L4b have the *same* graph, and only one of them recurses.

### What you can build now

Each pattern you just ran is a real product shape:

| You practiced | In the wild, that's | Start from |
| --- | --- | --- |
| Graph + router (L2a/L2b) | document pipelines, ETL-with-LLM-steps, review/approval chains, eval harnesses | this repo's L2b |
| Coordinator + `single_turn` team (L3a) | a support copilot with specialist teams, triage desks, multi-lens review | [marathon demo](https://github.com/cuppibla/adk-2-marathon-demo) mode 2 |
| `task` agents (L3b) | intake forms, booking flows, onboarding, KYC — any "collect then act" | [`22_agent_in_workflow`](https://github.com/cuppibla/adk-workflows-compared/tree/main/examples/22_agent_in_workflow) |
| Dynamic width/depth (L4a/L4b) | research agents, report generators, audit sweeps over unknown-sized inputs | [marathon demo](https://github.com/cuppibla/adk-2-marathon-demo) mode 3 |

### They compose

The three patterns are **not mutually exclusive**. A graph node can call a collaborative coordinator; a specialist can launch a dynamic workflow. Choose the right pattern per *part* of the problem — that's how you avoid turning every agent system into one giant prompt.

> 🏅 *Predictable work stays functions; clear rules become explicit routing; reasoning uses the model.*
> 🌟 *Let the LLM shape the work, but keep the boundaries in code.*
> 🎯 *Match the pattern to the shape of your problem.*

```
  \o/   You finished the marathon! 🏁🎉
   |    You now know all three ADK 2 orchestration patterns — and all three modes.
  / \   Go build something amazing — and keep the boundaries in code. 💪
```

🐾 *Made with love by Annie — happy building!*